In [2]:
import os
import glob
from datasets import load_dataset

# 诊断代码
general_path = "/DATA/disk2/yuhang/.cache/huggingface/hub/datasets--Mxode--IndustryCorpus-Subset-zh-en/snapshots/1e189ae582d2dadf1ab069f09b3fcebfa6ed4d5f"

print("=== 诊断信息 ===")

# 1. 检查目录是否存在
print(f"目录是否存在: {os.path.exists(general_path)}")

# 2. 列出所有parquet文件
parquet_files = glob.glob(f"{general_path}/*.parquet")
print(f"找到的parquet文件数量: {len(parquet_files)}")

# 3. 检查前几个文件的状态
for i, file_path in enumerate(parquet_files[:3]):  # 只检查前3个文件
    print(f"\n文件 {i+1}: {os.path.basename(file_path)}")
    print(f"  - 文件存在: {os.path.exists(file_path)}")
    print(f"  - 文件大小: {os.path.getsize(file_path) if os.path.exists(file_path) else 'N/A'} bytes")
    print(f"  - 是否为软链接: {os.path.islink(file_path)}")
    
    if os.path.islink(file_path):
        link_target = os.readlink(file_path)
        print(f"  - 链接目标: {link_target}")
        
        # 检查绝对路径的链接目标
        if not os.path.isabs(link_target):
            abs_target = os.path.join(os.path.dirname(file_path), link_target)
            print(f"  - 绝对链接目标: {abs_target}")
            print(f"  - 链接目标存在: {os.path.exists(abs_target)}")

# 4. 尝试用pandas直接读取一个文件
try:
    import pandas as pd
    if parquet_files:
        test_file = parquet_files[0]
        print(f"\n=== 尝试用pandas读取第一个文件 ===")
        df = pd.read_parquet(test_file)
        print(f"成功读取，数据形状: {df.shape}")
        print(f"列名: {list(df.columns)}")
        if len(df) > 0:
            print(f"第一行数据: {df.iloc[0].to_dict()}")
        else:
            print("文件为空")
except Exception as e:
    print(f"pandas读取失败: {e}")

=== 诊断信息 ===
目录是否存在: True
找到的parquet文件数量: 2

文件 1: validation.parquet
  - 文件存在: True
  - 文件大小: 701256828 bytes
  - 是否为软链接: True
  - 链接目标: ../../blobs/af9c194cc546debc9534dc8f27fd509b06b62d3322d29463fa6b41404286fcc7
  - 绝对链接目标: /DATA/disk2/yuhang/.cache/huggingface/hub/datasets--Mxode--IndustryCorpus-Subset-zh-en/snapshots/1e189ae582d2dadf1ab069f09b3fcebfa6ed4d5f/../../blobs/af9c194cc546debc9534dc8f27fd509b06b62d3322d29463fa6b41404286fcc7
  - 链接目标存在: True

文件 2: train.parquet
  - 文件存在: True
  - 文件大小: 6300919868 bytes
  - 是否为软链接: True
  - 链接目标: ../../blobs/4e1b41bfa7af02e945d8828041fef9d9dd5566e2105eec9b54bf956e42915b53
  - 绝对链接目标: /DATA/disk2/yuhang/.cache/huggingface/hub/datasets--Mxode--IndustryCorpus-Subset-zh-en/snapshots/1e189ae582d2dadf1ab069f09b3fcebfa6ed4d5f/../../blobs/4e1b41bfa7af02e945d8828041fef9d9dd5566e2105eec9b54bf956e42915b53
  - 链接目标存在: True

=== 尝试用pandas读取第一个文件 ===
成功读取，数据形状: (274802, 3)
列名: ['text', 'industry_type', 'text_length']
第一行数据: {'text': 'USDA Announces Firs

In [4]:
import os
from datasets import load_dataset

def resolve_symlinks(file_pattern):
    """解析软链接并返回实际文件路径"""
    import glob
    files = glob.glob(file_pattern)
    resolved_files = []
    
    for file_path in files:
        if os.path.islink(file_path):
            # 解析软链接
            link_target = os.readlink(file_path)
            if not os.path.isabs(link_target):
                # 相对路径，转换为绝对路径
                abs_target = os.path.join(os.path.dirname(file_path), link_target)
            else:
                abs_target = link_target
            
            if os.path.exists(abs_target):
                resolved_files.append(abs_target)
            else:
                print(f"警告: 软链接目标不存在: {abs_target}")
        else:
            resolved_files.append(file_path)
    
    return resolved_files

# 使用解析后的文件路径
general_path = "/DATA/disk2/yuhang/.cache/huggingface/hub/datasets--Mxode--IndustryCorpus-Subset-zh-en/snapshots/1e189ae582d2dadf1ab069f09b3fcebfa6ed4d5f"
resolved_files = resolve_symlinks(f"{general_path}/*.parquet")

print(f"解析后的文件数量: {len(resolved_files)}")

if resolved_files:
    ds = load_dataset(
        "parquet",
        data_files=resolved_files,
        split="train"
    )
    print(f"成功加载数据集，大小: {len(ds)}")
else:
    print("没有找到有效的parquet文件")

解析后的文件数量: 2


Generating train split: 2747888 examples [00:25, 106771.81 examples/s]


成功加载数据集，大小: 2747888


In [5]:
ds

Dataset({
    features: ['text', 'industry_type', 'text_length'],
    num_rows: 2747888
})

In [6]:
# 统计第一条数据的字段长度
first_data = ds[0]
text_length = len(first_data['text'])
text_length


3666

In [7]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B",
                                          use_fast = True)

In [8]:
def tokenize_function(examples):
    return tokenizer(examples['text'],
                     max_length=2048,
                     truncation=True,
                     padding="max_length")

tokenizer_dataset = ds.map(tokenize_function,
                                num_proc=40,
                                batched=True, 
                                remove_columns=['text']) #! 移除原始文本节省内存

Map (num_proc=40): 100%|██████████| 2747888/2747888 [03:47<00:00, 12056.11 examples/s]


In [9]:
tokenizer_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

In [12]:
tokenizer_dataset

Dataset({
    features: ['industry_type', 'text_length', 'input_ids', 'attention_mask'],
    num_rows: 2747888
})

In [13]:
tokenizer_dataset.save_to_disk("/DATA/disk2/yuhang/.cache/bit_brain_data/Annealing/IndustryCorpus-Subset-zh-en",
                               max_shard_size = "2024MB")

Saving the dataset (14/14 shards): 100%|██████████| 2747888/2747888 [00:18<00:00, 145162.05 examples/s]
